# Pixel Catch: DQN learns from pixels 🕹️
A fruit falls from the top of a 10×10 screen; move the basket left/right to catch it. The agent sees **only the pixels**, like DeepMind's 2013 Atari agent, and learns with **Deep Q-Networks**: a neural net estimates Q(s, a), a replay buffer breaks correlations, and a target network stabilises the TD targets.

Background: [[Q-Learning and Policy Gradients]], [[CNN]] · overview: [[Fun Projects]] · paper: [Mnih et al. 2013](https://arxiv.org/abs/1312.5602) · more games: [[RL Arcade]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import random, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); random.seed(0); np.random.seed(0)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
S = 10
class Catch:
    def reset(self):
        self.fruit = [0, random.randrange(S)]; self.basket = random.randrange(1, S - 1); return self.obs()
    def obs(self):
        img = np.zeros((S, S), np.float32); img[self.fruit[0], self.fruit[1]] = 1; img[S - 1, self.basket - 1:self.basket + 2] = 1; return img
    def step(self, a):                                                 # 0 left, 1 stay, 2 right
        self.basket = int(np.clip(self.basket + a - 1, 1, S - 2)); self.fruit[0] += 1
        if self.fruit[0] == S - 1: return self.obs(), (1.0 if abs(self.fruit[1] - self.basket) <= 1 else -1.0), True
        return self.obs(), 0.0, False
env = Catch(); plt.imshow(env.reset(), cmap="gray"); plt.title("what the agent sees"); plt.axis("off"); plt.show()

## 1. The TD loss\nFor a batch of transitions $(s,a,r,s',\text{done})$: target $y = r + \gamma\max_{a'}Q_{\text{target}}(s',a')\,(1-\text{done})$ (no gradient through it), prediction $Q(s,a)$. Use the Huber loss (`F.smooth_l1_loss`), as in the DQN paper.

In [ ]:
def td_loss(net, target_net, s, a, r, s2, done, gamma=0.95):
    # TODO 1: Q(s,a) from net, bootstrapped target from target_net, Huber loss
    raise NotImplementedError  # TODO 1

## 2. Replay buffer
Store transitions in a list; when it holds more than `capacity`, drop the **oldest**.

In [ ]:
class Replay:
    def __init__(self, capacity=5000): self.buf, self.capacity = [], capacity
    def push(self, transition):
        # TODO 2: append, evict the oldest when over capacity
        raise NotImplementedError  # TODO 2
    def sample(self, n):
        s, a, r, s2, d = zip(*random.sample(self.buf, n))
        return torch.tensor(np.array(s)), torch.tensor(a), torch.tensor(r, dtype=torch.float32), torch.tensor(np.array(s2)), torch.tensor(d, dtype=torch.float32)

In [ ]:
def _t_dqn():
    rb = Replay(3)
    for i in range(5): rb.push(i)
    net = lambda x: torch.tensor([[1.0, 2.0], [0.0, 5.0]]); tgt = lambda x: torch.tensor([[3.0, 1.0], [2.0, 4.0]])
    l = td_loss(net, tgt, None, torch.tensor([1, 0]), torch.tensor([1.0, -1.0]), None, torch.tensor([0.0, 1.0]), gamma=0.5)
    # targets: 1 + 0.5*3 = 2.5 (pred 2.0, diff .5 → .125) ; -1 (done) (pred 0, diff 1 → .5)  → mean .3125
    return rb.buf == [2, 3, 4] and abs(l.item() - 0.3125) < 1e-6
check("DQN parts", _t_dqn, "Buffer keeps the newest 3; TD targets 2.5 and −1 (done) give Huber loss 0.3125.")

def make_net(): return nn.Sequential(nn.Flatten(), nn.Linear(S * S, 64), nn.ReLU(), nn.Linear(64, 3))
def evaluate(net, n=200):
    wins = 0
    for _ in range(n):
        o, done = env.reset(), False
        while not done:
            with torch.no_grad(): a = int(net(torch.tensor(o)[None]).argmax())
            o, r, done = env.step(a)
        wins += r > 0
    return wins / n

if ready("DQN parts"):
    net, target_net = make_net(), make_net(); target_net.load_state_dict(net.state_dict())
    opt = torch.optim.Adam(net.parameters(), 1e-3); rb = Replay(); curve = []; step = 0
    print(f"random agent catches {np.mean([env.reset() is not None and (lambda: [env.step(random.randrange(3)) for _ in range(S - 1)][-1][1] > 0)() for _ in range(500)]):.0%}")
    for episode in range(800):
        o, done = env.reset(), False; eps = max(0.05, 1 - episode / 400)
        while not done:
            with torch.no_grad(): a = random.randrange(3) if random.random() < eps else int(net(torch.tensor(o)[None]).argmax())
            o2, r, done = env.step(a); rb.push((o, a, r, o2, float(done))); o = o2; step += 1
            if len(rb.buf) >= 256:
                loss = td_loss(net, target_net, *rb.sample(64)); opt.zero_grad(); loss.backward(); opt.step()
            if step % 200 == 0: target_net.load_state_dict(net.state_dict())
        if episode % 100 == 99: curve.append((episode + 1, evaluate(net))); print(f"episode {episode + 1}: catch rate {curve[-1][1]:.0%} (ε = {eps:.2f})")
    check("catches from pixels", lambda: curve[-1][1] >= 0.9, "After 800 episodes the DQN should catch ≥ 90 % of fruit.")

<details><summary>▶ Solution</summary>

```python
def td_loss(net, target_net, s, a, r, s2, done, gamma=0.95):
    with torch.no_grad():
        next_q = target_net(s2).max(1).values
        target = r + gamma * next_q * (1 - done)
    q = net(s).gather(1, a.unsqueeze(1)).squeeze(1)
    return F.smooth_l1_loss(q, target)
```

```python
    def push(self, transition):
        self.buf.append(transition)
        if len(self.buf) > self.capacity:
            self.buf.pop(0)
```
</details>

## Stretch goals
- Remove the target network (use `net` for targets too). Does learning become unstable?
- Replace the MLP with a small CNN and make the screen 20×20.
- Two fruits at once, or a fruit that drifts sideways: does the agent still cope?